# Beginner Workshop: Single-Cell RNA-seq Analysis with Scanpy
## PBMC 3k Tutorial — Basics & Visualization

Welcome to this hands-on workshop on single-cell RNA sequencing (scRNA-seq) analysis using **Scanpy** and other tools from the [scverse](https://scverse.org/) ecosystem.

### Workshop overview
| Section | Topic |
|---------|-------|
| 0 | Setup and imports |
| 1 | The AnnData data structure |
| 2 | Loading PBMC data |
| 3 | Quality control (QC) |
| 4 | Doublet detection |
| 5 | Preprocessing (normalization, HVGs, scaling) |
| 6 | Cell cycle scoring |
| 7 | Dimensionality reduction (PCA) |
| 8 | Clustering (Leiden) |
| 9 | Embeddings and visualization (UMAP, t-SNE) |
| 10 | Visualization gallery |
| 11 | Marker gene detection |
| 12 | Cell type annotation |
| 13 | Workshop exercises |

### Dataset
We use the [PBMC 3k](https://support.10xgenomics.com/single-cell-gene-expression/datasets/1.1.0/pbmc3k) dataset — 2,700 peripheral blood mononuclear cells from a healthy donor sequenced on the 10x Chromium v1 platform. This is the standard Scanpy tutorial dataset.

> **Prerequisites:** basic Python knowledge. No prior scRNA-seq experience required.


## 0. Setup

Install the required packages if they are not already available:
```bash
pip install scanpy anndata matplotlib seaborn scrublet
```

The key packages we use:
- **scanpy** — single-cell analysis in Python
- **anndata** — annotated data matrix (the core data structure)
- **matplotlib / seaborn** — plotting
- **scrublet** — doublet detection


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata as ad

# Reproducibility
import random
random.seed(0)
np.random.seed(0)

# Scanpy settings
sc.settings.verbosity = 3          # print progress messages (0–3)
sc.settings.n_jobs = 1             # use 1 CPU core; increase for large datasets
sc.set_figure_params(dpi=100, facecolor='white', figsize=(6, 5))

print('scanpy version:', sc.__version__)
print('anndata version:', ad.__version__)


## 1. The AnnData Data Structure

Almost everything in Scanpy revolves around an **AnnData** object.  Think of it as a smart spreadsheet that holds your cells and genes together with all associated metadata.

```
              genes (var)
          ┌────────────────────────┐
   cells  │                        │
   (obs)  │    X  (expression      │
          │       matrix)          │
          └────────────────────────┘
```

| Slot | What it holds |
|------|---------------|
| `adata.X` | count / expression matrix (cells × genes) |
| `adata.obs` | cell metadata (e.g., QC metrics, cluster labels) |
| `adata.var` | gene metadata (e.g., HVG flags, mean expression) |
| `adata.obsm` | cell-level embeddings (e.g., PCA, UMAP coordinates) |
| `adata.uns` | unstructured data (e.g., color palettes, neighbor graph) |
| `adata.layers` | additional matrices (e.g., raw counts, normalized) |
| `adata.raw` | snapshot of the expression matrix before filtering |

> **Workshop question:** Why do you think it is useful to keep all of this information in one object?


## 2. Loading PBMC Data

Scanpy ships a helper function `sc.datasets.pbmc3k()` that downloads the PBMC 3k dataset automatically.

The raw data are **UMI counts** produced by Cell Ranger from 10x Genomics chromium v1 chemistry.


In [ ]:
adata = sc.datasets.pbmc3k()
adata.var_names_make_unique()  # ensure gene names are unique

print('Shape:', adata.shape)   # (cells, genes)
print(adata)


In [ ]:
# Inspect the first few cells
print('obs (cell metadata):'); print(adata.obs.head())
print()
print('var (gene metadata):'); print(adata.var.head())


In [ ]:
# Store the raw (pre-QC) counts for later reference
adata.layers['counts'] = adata.X.copy()


## 3. Quality Control (QC)

Before any analysis, we need to remove **low-quality cells** and **empty droplets**.

The three most commonly used QC metrics are:
1. **Number of genes per cell** — low values suggest empty droplets; extremely high values may indicate doublets.
2. **Total UMI counts per cell** — closely related to gene count.
3. **Fraction of mitochondrial counts** — high mitochondrial fraction suggests a damaged or dying cell (its cytoplasmic RNA has leaked out, leaving mostly mt-RNA).

For PBMC data, mitochondrial genes start with `MT-`.


In [ ]:
# Flag mitochondrial genes
adata.var['mt'] = adata.var_names.str.startswith('MT-')

# Flag ribosomal genes (optional extra filter)
adata.var['ribo'] = adata.var_names.str.startswith(('RPS', 'RPL'))

# Calculate QC metrics
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=['mt', 'ribo'],
    percent_top=None,
    log1p=False,
    inplace=True
)

print('QC columns added to obs:')
print([c for c in adata.obs.columns])


In [ ]:
# ── Violin plots of QC metrics ────────────────────────────────────────
sc.pl.violin(
    adata,
    ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo'],
    jitter=0.4,
    multi_panel=True,
    groupby=None,
)


In [ ]:
# ── Scatter plots: spot outlier cells ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt', ax=axes[0], show=False)
axes[0].axhline(5, color='red', linestyle='--', label='5% MT threshold')
axes[0].legend()

sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', ax=axes[1], show=False)
axes[1].axhline(2500, color='red', linestyle='--', label='2500 gene threshold')
axes[1].legend()

plt.tight_layout()
plt.show()


### 3.1 Filtering

Apply thresholds based on the plots above. The commonly used PBMC3k tutorial thresholds are:
- Remove cells with **< 200 genes** (likely empty droplets)
- Remove cells with **> 2500 genes** (likely doublets)
- Remove cells with **> 5% mitochondrial reads** (likely dying cells)
- Remove genes expressed in **fewer than 3 cells**

> **Workshop question:** How would you determine appropriate thresholds for a new dataset?


In [ ]:
print(f'Before filtering: {adata.n_obs} cells, {adata.n_vars} genes')

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

adata = adata[adata.obs.n_genes_by_counts < 2500, :].copy()
adata = adata[adata.obs.pct_counts_mt < 5, :].copy()

print(f'After filtering:  {adata.n_obs} cells, {adata.n_vars} genes')


## 4. Doublet Detection with Scrublet

**Doublets** are droplets that accidentally captured two cells. They can create false clusters and confound downstream analysis.

[Scrublet](https://github.com/swolock/scrublet) simulates synthetic doublets by combining pairs of real cells and scores each observed cell by its similarity to those simulated doublets.

> **Note:** Scrublet works on **raw counts**. We use the stored `counts` layer.


In [ ]:
try:
    import scrublet as scr

    # Run scrublet on raw counts
    scrub = scr.Scrublet(adata.layers['counts'])
    doublet_scores, predicted_doublets = scrub.scrub_doublets()

    adata.obs['doublet_score'] = doublet_scores
    adata.obs['predicted_doublet'] = predicted_doublets

    scrub.plot_histogram()
    plt.show()

    n_doublets = predicted_doublets.sum()
    print(f'Predicted doublets: {n_doublets} / {adata.n_obs} ({100*n_doublets/adata.n_obs:.1f}%)')

    # Remove predicted doublets
    adata = adata[~adata.obs['predicted_doublet']].copy()
    print(f'After doublet removal: {adata.n_obs} cells')

except ImportError:
    print('scrublet not installed — skipping doublet detection.')
    print('Install with: pip install scrublet')


## 5. Preprocessing

### 5.1 Normalization

Cell Ranger counts vary in depth between cells (some cells were sequenced more deeply). We apply **total-count normalization** to make cells comparable:

1. Scale each cell to a total of **10,000 counts** ("library-size normalization")
2. Apply **log1p** transformation: `log(x + 1)` — compresses the dynamic range and makes the data more normally distributed


In [ ]:
# Normalize to 10,000 counts per cell
sc.pp.normalize_total(adata, target_sum=1e4)

# Log-transform
sc.pp.log1p(adata)

# Save a snapshot of the normalized, log-transformed matrix as '.raw'
# This is used later for visualizing gene expression
adata.raw = adata

print('Normalization complete.')
print('adata.raw stores the full gene set for visualization.')


### 5.2 Highly Variable Genes (HVGs)

Most genes show little variation across cells and add noise. We select the **highly variable genes** — those that vary more than expected by chance — to focus downstream analyses.

By default, Scanpy uses the Seurat v1 method (Flynn et al.): genes are binned by mean expression, and dispersion (variance/mean) is normalized within each bin.


In [ ]:
sc.pp.highly_variable_genes(
    adata,
    min_mean=0.0125,
    max_mean=3,
    min_disp=0.5
)

print(f'Highly variable genes: {adata.var.highly_variable.sum()} / {adata.n_vars}')

sc.pl.highly_variable_genes(adata)


### 5.3 Regression and Scaling

**Regress out** technical sources of variation (total counts, mitochondrial fraction) so they do not dominate the PCA.

**Scaling** each gene to unit variance (z-score) ensures all genes contribute equally to PCA.


In [ ]:
# Subset to HVGs (keep .raw for full-gene visualization)
adata = adata[:, adata.var.highly_variable].copy()

# Regress out confounders
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])

# Scale to unit variance; clip at 10 SDs to reduce the effect of extreme outliers
sc.pp.scale(adata, max_value=10)

print('Preprocessing complete.')
print('adata shape after HVG subset:', adata.shape)


## 6. Cell Cycle Scoring

Variation in the cell cycle is a major source of technical confounding in scRNA-seq data. We can score each cell for its **S-phase** and **G2/M-phase** expression signatures using known marker genes.

Scanpy provides the `sc.tl.score_genes_cell_cycle` function. We use the classic Tirosh et al. 2016 gene lists (included in Scanpy as `sc.datasets.pbmc3k_processed` convenience or available via `scanpy.datasets`).

> **Note:** For this PBMC dataset the cell-cycle effect is modest; it is more prominent in actively cycling tumor or stem-cell datasets.


In [ ]:
# Canonical S-phase and G2/M-phase genes (Tirosh et al. 2016)
s_genes = [
    'MCM5', 'PCNA', 'TYMS', 'FEN1', 'MCM2', 'MCM4', 'RRM1', 'UNG',
    'GINS2', 'MCM6', 'CDCA7', 'DTL', 'PRIM1', 'UHRF1', 'MLF1IP',
    'HELLS', 'RFC2', 'RPA2', 'NASP', 'RAD51AP1', 'GMNN', 'WDR76',
    'SLBP', 'CCNE2', 'UBR7', 'POLD3', 'MSH2', 'ATAD2', 'RAD51',
    'RRM2', 'CDC45', 'CDC6', 'EXO1', 'TIPIN', 'DSCC1', 'BLM',
    'CASP8AP2', 'USP1', 'CLSPN', 'POLA1', 'CHAF1B', 'BRIP1', 'E2F8'
]

g2m_genes = [
    'HMGB2', 'CDK1', 'NUSAP1', 'UBE2C', 'BIRC5', 'TPX2', 'TOP2A',
    'NDC80', 'CKS2', 'NUF2', 'CKS1B', 'MKI67', 'TMPO', 'CENPF',
    'TACC3', 'FAM64A', 'SMC4', 'CCNB2', 'CKAP2L', 'CKAP2', 'AURKB',
    'BUB1', 'KIF11', 'ANP32E', 'TUBB4B', 'GTSE1', 'KIF20B', 'HJURP',
    'CDCA3', 'HN1', 'CDC20', 'TTK', 'CDC25C', 'KIF2C', 'RANGAP1',
    'NCAPD2', 'DLGAP5', 'CDCA2', 'CDCA8', 'ECT2', 'KIF23', 'HMMR',
    'AURKA', 'PSRC1', 'ANLN', 'LBR', 'CKAP5', 'CENPE', 'CTCF',
    'NEK2', 'G2E3', 'GAS2L3', 'CBX5', 'CENPA'
]

# Only score genes that are present in our HVG-filtered dataset
s_genes_use   = [g for g in s_genes   if g in adata.var_names]
g2m_genes_use = [g for g in g2m_genes if g in adata.var_names]

if s_genes_use and g2m_genes_use:
    sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes_use, g2m_genes=g2m_genes_use)
    print('Cell cycle phase distribution:')
    print(adata.obs['phase'].value_counts())
else:
    print('Not enough cell cycle genes found in HVG subset; skipping.')


## 7. Principal Component Analysis (PCA)

PCA reduces the ~2000 HVG dimensions to a smaller number of **principal components** (PCs) that capture the most variation in the data.

We will:
1. Compute 50 PCs
2. Inspect the **elbow plot** to decide how many PCs to use downstream
3. Visualize cells in PC space


In [ ]:
sc.tl.pca(adata, svd_solver='arpack', n_comps=50)

# Elbow plot — variance explained per PC
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50)


In [ ]:
# Visualize cells in PC1 vs PC2
sc.pl.pca(
    adata,
    color=['total_counts', 'pct_counts_mt'],
    ncols=2,
    show=True
)


In [ ]:
# If cell cycle scoring was done, color by phase
if 'phase' in adata.obs.columns:
    sc.pl.pca(adata, color='phase')


> **Workshop question:** How many PCs do you think explain most of the meaningful biological variation? Look at the elbow plot.


## 8. Clustering with the Leiden Algorithm

We cluster cells using a graph-based approach:
1. **Build a k-nearest-neighbor (kNN) graph** in PC space
2. **Detect communities** in the graph using the Leiden algorithm (an improvement over Louvain)

The `resolution` parameter controls granularity: higher values → more clusters.


In [ ]:
# Build neighborhood graph using the first 40 PCs
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

# Run UMAP first (we'll need it for visualization)
sc.tl.umap(adata)

# Leiden clustering at multiple resolutions
for res in [0.3, 0.5, 1.0]:
    key = f'leiden_res{res}'
    sc.tl.leiden(adata, resolution=res, key_added=key)
    print(f'Leiden res={res}: {adata.obs[key].nunique()} clusters')


## 9. Embeddings and Visualization

### 9.1 UMAP

UMAP (Uniform Manifold Approximation and Projection) projects cells into 2D space while preserving local neighborhood structure.

> **Important:** UMAP does *not* preserve global distances — do not over-interpret the relative position of distant clusters.


In [ ]:
sc.pl.umap(
    adata,
    color=['leiden_res0.3', 'leiden_res0.5', 'leiden_res1.0'],
    ncols=3,
    legend_loc='on data',
    title=['Leiden (res=0.3)', 'Leiden (res=0.5)', 'Leiden (res=1.0)']
)


In [ ]:
# Color by QC metrics on UMAP
sc.pl.umap(
    adata,
    color=['total_counts', 'n_genes_by_counts', 'pct_counts_mt'],
    ncols=3,
    title=['Total counts', 'Genes per cell', '% MT reads']
)


In [ ]:
# Color by known PBMC markers on UMAP
marker_genes_umap = {
    'CD3D':  'T cells',
    'CD19':  'B cells',
    'CD14':  'Monocytes',
    'GNLY':  'NK cells',
    'FCER1A':'Dendritic cells',
    'PPBP':  'Megakaryocytes',
}
genes_present = [g for g in marker_genes_umap if g in adata.raw.var_names]
sc.pl.umap(
    adata,
    color=genes_present,
    use_raw=True,
    ncols=3,
    title=[f'{g} ({marker_genes_umap[g]})' for g in genes_present]
)


### 9.2 t-SNE

t-SNE is another 2D projection technique. It tends to create very compact clusters but is less scalable and slower than UMAP for large datasets.


In [ ]:
sc.tl.tsne(adata, use_rep='X_pca', learning_rate='auto')
sc.pl.tsne(adata, color='leiden_res0.5', legend_loc='on data')


## 10. Visualization Gallery

Scanpy provides many plot types for exploring gene expression and cluster relationships. We show the main ones here.

We use **res=0.5** clusters for all visualizations below and a set of canonical PBMC marker genes.


In [ ]:
# Canonical PBMC marker gene sets (for visualization)
pbmc_markers = {
    'CD4 T': ['IL7R', 'CCR7', 'S100A4'],
    'CD8 T': ['CD8A', 'CD8B'],
    'B':     ['CD19', 'MS4A1', 'CD79A'],
    'NK':    ['GNLY', 'NKG7', 'GZMB'],
    'Mono':  ['CD14', 'LYZ', 'CST3'],
    'DC':    ['FCER1A', 'CST3'],
    'Plt':   ['PPBP'],
}

# Flatten to a list of unique genes present in the dataset
all_markers = [g for genes in pbmc_markers.values() for g in genes]
all_markers = [g for g in dict.fromkeys(all_markers) if g in adata.raw.var_names]
print('Marker genes available:', all_markers)


In [ ]:
# ── Dot plot ──────────────────────────────────────────────────────────
# Shows the fraction of cells expressing a gene (dot size)
# and the mean expression level (dot color) per cluster
sc.pl.dotplot(
    adata,
    var_names=pbmc_markers,
    groupby='leiden_res0.5',
    dendrogram=True,
    use_raw=True,
    standard_scale='var',
    title='Marker gene expression by cluster',
)


In [ ]:
# ── Violin plot ───────────────────────────────────────────────────────
# Shows distribution of expression per cluster
sc.pl.violin(
    adata,
    keys=all_markers[:6],   # show first 6 markers
    groupby='leiden_res0.5',
    rotation=45,
    use_raw=True,
)


In [ ]:
# ── Heatmap ───────────────────────────────────────────────────────────
# Shows mean expression per cluster as a color grid
sc.pl.heatmap(
    adata,
    var_names=pbmc_markers,
    groupby='leiden_res0.5',
    use_raw=True,
    standard_scale='var',
    dendrogram=True,
    cmap='viridis',
)


In [ ]:
# ── Matrix plot ───────────────────────────────────────────────────────
# Similar to heatmap but averages within groups
sc.pl.matrixplot(
    adata,
    var_names=pbmc_markers,
    groupby='leiden_res0.5',
    use_raw=True,
    standard_scale='var',
    dendrogram=True,
    cmap='RdBu_r',
)


In [ ]:
# ── Stacked violin plot ───────────────────────────────────────────────
sc.pl.stacked_violin(
    adata,
    var_names=pbmc_markers,
    groupby='leiden_res0.5',
    use_raw=True,
    dendrogram=True,
)


In [ ]:
# ── Tracksplot ────────────────────────────────────────────────────────
# Shows expression as horizontal bars for each cell, sorted by cluster
sc.pl.tracksplot(
    adata,
    var_names=all_markers[:8],
    groupby='leiden_res0.5',
    use_raw=True,
)


## 11. Marker Gene Detection

To understand what each cluster represents biologically, we perform **differential gene expression** to find genes that are highly expressed in one cluster compared to all others (1-vs-rest).

Scanpy offers several statistical tests:
- **`wilcoxon`** — non-parametric, robust, recommended for most cases
- **`t-test`** — faster but assumes normality
- **`logreg`** — logistic regression, useful when many clusters


In [ ]:
# Use .raw so we have all genes (not just HVGs)
sc.tl.rank_genes_groups(
    adata,
    groupby='leiden_res0.5',
    method='wilcoxon',
    use_raw=True,
    n_genes=25,
)

# Overview plot
sc.pl.rank_genes_groups(adata, n_genes=15, sharey=False)


In [ ]:
# Table of top marker genes per cluster
marker_df = sc.get.rank_genes_groups_df(adata, group=None)
print(marker_df.head(20))


In [ ]:
# Dotplot of top 3 marker genes per cluster
sc.pl.rank_genes_groups_dotplot(
    adata,
    n_genes=3,
    use_raw=True,
    standard_scale='var',
)


In [ ]:
# Violin plot of top markers
sc.pl.rank_genes_groups_violin(
    adata,
    groups=['0', '1', '2'],
    n_genes=5,
    use_raw=True,
)


## 12. Cell Type Annotation

Based on the marker genes from section 11 and published PBMC signatures, we can annotate each cluster with a cell type label.

### Canonical PBMC cell type markers

| Cell type | Key markers |
|-----------|------------|
| CD4 T cells | IL7R, CCR7, S100A4 |
| CD8 T cells | CD8A, CD8B |
| B cells | CD19, MS4A1, CD79A |
| NK cells | GNLY, NKG7, GZMB |
| CD14+ Monocytes | CD14, LYZ, CST3 |
| FCGR3A+ Monocytes | FCGR3A, MS4A7 |
| Dendritic cells | FCER1A, CST3 |
| Megakaryocytes | PPBP |

> **Workshop exercise:** Look at your cluster marker genes and the canonical markers above. Can you identify what cell type each cluster is?


In [ ]:
# Example annotation mapping (adjust based on your clustering results)
# NOTE: cluster numbers vary between runs — inspect your own marker output!
cluster_to_celltype = {
    '0': 'CD4 T',
    '1': 'CD14+ Mono',
    '2': 'CD4 T',
    '3': 'NK / CD8 T',
    '4': 'B',
    '5': 'CD8 T',
    '6': 'FCGR3A+ Mono',
    '7': 'NK',
    '8': 'DC',
    '9': 'Platelet',
}

# Map to a new column
adata.obs['cell_type'] = (
    adata.obs['leiden_res0.5']
         .map(cluster_to_celltype)
         .fillna('Unknown')
         .astype('category')
)

sc.pl.umap(
    adata,
    color='cell_type',
    legend_loc='on data',
    title='Annotated PBMC cell types',
    frameon=False,
)


In [ ]:
# Final dotplot with annotated cell types
sc.pl.dotplot(
    adata,
    var_names=pbmc_markers,
    groupby='cell_type',
    use_raw=True,
    standard_scale='var',
    dendrogram=True,
)


In [ ]:
# Bar chart of cell type proportions
cell_counts = adata.obs['cell_type'].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
cell_counts.plot.bar(ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Cell type')
ax.set_ylabel('Number of cells')
ax.set_title('Cell type composition of PBMC 3k dataset')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 13. Save the Processed Object

Save the annotated AnnData to disk for reproducibility and downstream analysis.


In [ ]:
adata.write('pbmc3k_workshop_annotated.h5ad')
print('Saved to pbmc3k_workshop_annotated.h5ad')


## 14. Workshop Exercises

Try these exercises to deepen your understanding:

### Beginner
1. **Change QC thresholds** — Re-run with `pct_counts_mt < 10` instead of 5. How does it affect cell numbers and cluster composition?
2. **Change Leiden resolution** — Try `resolution=2.0`. How many clusters do you get? Do the new sub-clusters make biological sense?
3. **Color the UMAP by additional genes** — Look up another PBMC marker gene and visualize it on the UMAP.

### Intermediate
4. **Add ribosomal QC** — Filter cells with `pct_counts_ribo > 50`. Does this change the results?
5. **Compare UMAP and t-SNE** — What differences do you notice in how the clusters are arranged?
6. **Regress out cell cycle scores** — Add `S_score` and `G2M_score` to the `regress_out` call. Does this change the PCA and UMAP?

### Advanced
7. **Try different HVG parameters** — Use `flavor='seurat_v3'` in `sc.pp.highly_variable_genes`. How does this change the selected genes?
8. **Compute a trajectory** — Install `scvelo` or `diffmap` and explore pseudotime ordering of T cells.
9. **Batch correction** — Download the PBMC 68k dataset and combine with PBMC 3k. Run [Harmony](https://github.com/immunogenomics/harmony) or [scanorama](https://github.com/brianhie/scanorama) to correct batch effects.

### Reflection questions
- Why do we store `adata.raw` before subsetting to HVGs?
- What happens if you skip the `regress_out` step?
- Why is it important to choose the number of PCs carefully for the neighbor graph?
